# 3. BERT y comparación

**Responsable: Sara.**

El extractor BERT y su prueba con diez textos proceden del cuaderno original
de Mónica. Aquí se ha añadido selección de GPU y guardado compatible con el cuaderno de Izan.
La extracción completa, el fine-tuning y el análisis final están pendientes.

Ejecutar desde la carpeta del proyecto. En Colab, proporcionar los Excel,
funciones.py y el mismo resultados/particiones.csv antes de empezar.


In [ ]:
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from funciones import RESULTADOS, CACHE, guardar_representacion


In [ ]:
from funciones import cargar_datos, preparar_datos, cargar_o_crear_particiones

projects, scivoc = cargar_datos()
df = preparar_datos(projects, scivoc)
conjuntos = cargar_o_crear_particiones(df)
df_train = conjuntos["train"]
df_val = conjuntos["val"]
df_test = conjuntos["test"]


## BERT congelado

Usamos el resumen original, sin el preprocesamiento de TF-IDF.
Mantenemos bert-base-uncased, media de tokens sin padding y truncamiento
a 512 tokens. La media incluye los tokens especiales, como en la base inicial.
Esta primera carga descarga los pesos si todavía no están en el ordenador.


In [ ]:
modelo_nombre = "bert-base-uncased"
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(modelo_nombre)
modelo_bert = AutoModel.from_pretrained(modelo_nombre).to(dispositivo)
modelo_bert.requires_grad_(False)
modelo_bert.eval()
print("Dispositivo:", dispositivo)


In [ ]:
textos = {
    nombre: parte["summary"].fillna("").astype(str).tolist()
    for nombre, parte in conjuntos.items()
}
for nombre, lista in textos.items():
    print(nombre, len(lista))


In [ ]:
def obtener_embeddings_bert(textos, batch_size=4):
    embeddings = []
    for i in tqdm(range(0, len(textos), batch_size), desc="Extrayendo BERT"):
        entradas = tokenizer(
            textos[i:i + batch_size], padding=True, truncation=True,
            max_length=512, return_tensors="pt",
        )
        entradas = {k: v.to(dispositivo) for k, v in entradas.items()}
        with torch.no_grad():
            salida = modelo_bert(**entradas).last_hidden_state
            mascara = entradas["attention_mask"].unsqueeze(-1).to(salida.dtype)
            vector = (salida * mascara).sum(dim=1) / mascara.sum(dim=1).clamp(min=1)
        embeddings.append(vector.cpu().numpy())
    if not embeddings:
        return np.empty((0, modelo_bert.config.hidden_size), dtype=np.float32)
    return np.vstack(embeddings)


## Prueba con diez resúmenes


In [ ]:
X_prueba_bert = obtener_embeddings_bert(textos["train"][:10], batch_size=2)
print("Dimensiones de la prueba:", X_prueba_bert.shape)


## Extracción completa

Cambiar GENERAR_TODO a True cuando se quiera ejecutar el corpus completo.
Los vectores se guardan fuera de GitHub en datos/cache/bert.joblib.
Izan podrá cargarlos con cargar_representacion("bert", conjuntos).


In [ ]:
GENERAR_TODO = False

if GENERAR_TODO:
    batch_size = 8 if dispositivo.type == "cuda" else 4
    matrices_bert = {
        nombre: obtener_embeddings_bert(lista, batch_size=batch_size)
        for nombre, lista in textos.items()
    }
    ruta = guardar_representacion("bert", matrices_bert, conjuntos)
    print("Vectores guardados en:", ruta)
else:
    print("Extracción completa pendiente: activar GENERAR_TODO para ejecutarla.")


**Pendiente:** medir qué porcentaje de resúmenes supera los 512 tokens y
explicar qué información puede perderse por el truncamiento.


## Pendiente: fine-tuning

Preparar un Transformer para clasificación con Hugging Face y actualizar sus
pesos usando las etiquetas. Mantener el resumen original y las particiones.
Convertir los códigos de categoría a índices consecutivos y guardar el mapeo.
Anotar modelo, tokenización, longitud máxima, épocas, batch size, tasa de
aprendizaje y selección mediante validación. El test se usa al final.

No basta con entrenar una capa sobre los vectores congelados: este experimento
tiene que actualizar los parámetros del Transformer.


In [ ]:
# PENDIENTE: entrenamiento de clasificación con fine-tuning.


## Comparación de resultados disponibles

Esta tabla se rellena al ejecutar los experimentos. Separar validación y test,
nivel 1 y nivel 2; no mezclar resultados iniciales con los definitivos.


In [ ]:
metricas = pd.read_csv(RESULTADOS / "metricas.csv")
if metricas.empty:
    print("Todavía no hay métricas guardadas en esta copia del proyecto.")
else:
    display(metricas.sort_values(
        ["nivel", "conjunto", "macro_f1"], ascending=[True, True, False]
    ))


## Pendiente: evaluación final y nivel 1

Reunir resultados de Izan y del fine-tuning. Mostrar macro-F1, accuracy y métricas
por clase; incluir curvas de aprendizaje o validación y la matriz de confusión
completa de la mejor configuración.

Comparar el ranking de las tres representaciones en los niveles 1 y 2.
Si el fine-tuning es el mejor modelo global, incluir su repetición en nivel 1.


In [ ]:
# PENDIENTE: tablas y figuras de la evaluación final.


## Pendiente: incertidumbre y errores

Establecer qué diferencia de rendimiento es suficientemente clara con nuestro
test y aplicar ese criterio a las conclusiones. Usar predicciones de los
mismos proyectos para comparar modelos.

Revisar ejemplos, clases pequeñas y confusiones entre categorías hermanas.
Se puede comprobar si la predicción era otra de las categorías válidas del
proyecto que descartamos al elegir una sola etiqueta.


In [ ]:
# PENDIENTE: análisis de incertidumbre y ejemplos de errores.


## Para cerrar el trabajo

Integrar el texto que escriban Mónica e Izan en el README, incorporar todas las figuras
usadas en la discusión y revisar las conclusiones con el grupo.
Completar las contribuciones reales, las referencias y las diapositivas.
Comprobar que otra persona puede repetir el trabajo desde esta carpeta.
